In [ ]:
# AI experiment pipeline notebook (single-script style)
# ----------------------------------------------------
# This script:
# 1) Loads prompt data from CSV
# 2) Loads baseline/context templates
# 3) Calls GPT and Claude for each row (or mock mode)
# 4) Saves outputs to data/results.csv (with safety checks)

from pathlib import Path
import time
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic


# Step 0: Runtime configuration (safe defaults)
MOCK_MODE = True
RUN_LIMIT = 1
RUN_FULL_EXPERIMENT = False

OPENAI_MODEL_NAME = "gpt-4o-mini"
CLAUDE_MODEL_NAME = "claude-3-haiku-20240307"


# Step 1: Safer project path resolution
# Works whether kernel starts in repo root or notebooks/
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "prompts.csv").exists():
        return cwd
    if (cwd.parent / "data" / "prompts.csv").exists():
        return cwd.parent
    raise FileNotFoundError("Could not locate project root containing data/prompts.csv")


PROJECT_ROOT = resolve_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "prompts.csv"
BASELINE_TEMPLATE_PATH = PROJECT_ROOT / "prompts" / "baseline.txt"
CONTEXT_TEMPLATE_PATH = PROJECT_ROOT / "prompts" / "context.txt"
RESULTS_PATH = PROJECT_ROOT / "data" / "results.csv"


# Step 2: Load .env for API keys
# OPENAI_API_KEY and ANTHROPIC_API_KEY should be in .env for real runs.
load_dotenv(PROJECT_ROOT / ".env")


# Step 3: Load input data and templates
prompts_df = pd.read_csv(DATA_PATH)

with open(BASELINE_TEMPLATE_PATH, "r", encoding="utf-8") as f:
    baseline_template = f.read()

with open(CONTEXT_TEMPLATE_PATH, "r", encoding="utf-8") as f:
    context_template = f.read()


# Step 4: Normalize placeholders and format prompts
# Current files use {candidate_information}; code uses {candidate_info}.
baseline_template = baseline_template.replace("{candidate_information}", "{candidate_info}")
context_template = context_template.replace("{candidate_information}", "{candidate_info}")


def format_prompt(
    template: str,
    candidate_info: str,
    job_description: str,
    task_instruction: str,
) -> str:
    """Fill one prompt template with row values, ensuring task instruction is included."""
    prompt = template.format(
        candidate_info=candidate_info,
        job_description=job_description,
        task_instruction=task_instruction,
    )

    if "{task_instruction}" not in template and str(task_instruction).strip():
        prompt += f"\n\nTask instruction:\n{task_instruction}"

    return prompt


# Step 5: API clients (lazy init)
openai_client = None
anthropic_client = None


# Step 6: Helpers

def is_error_output(text: str) -> bool:
    return str(text).startswith("ERROR (")


def get_candidate_value(row: pd.Series) -> str:
    return row.get("candidate_information", row.get("candidate_info", ""))


def get_task_instruction_value(row: pd.Series) -> str:
    return row.get("task_instruction", "")


def mock_response(model_label: str, prompt_type: str, prompt_id: str, prompt: str) -> str:
    preview = prompt.replace("\n", " ")[:120]
    return f"MOCK ({model_label}/{prompt_type}/{prompt_id}): synthetic response generated. Prompt preview: {preview}"


# Step 7: Model call helpers with basic error handling + one retry

def call_gpt(prompt: str, prompt_type: str, prompt_id: str) -> str:
    """Call OpenAI GPT model and return text output."""
    global openai_client

    if MOCK_MODE:
        return mock_response("gpt", prompt_type, prompt_id, prompt)

    if openai_client is None:
        try:
            openai_client = OpenAI()
        except Exception as e:
            return f"ERROR (GPT): {e}"

    for attempt in range(2):
        try:
            response = openai_client.responses.create(
                model=OPENAI_MODEL_NAME,
                input=prompt,
            )
            return response.output_text
        except Exception as e:
            if attempt == 0:
                time.sleep(1)
            else:
                return f"ERROR (GPT): {e}"


def call_claude(prompt: str, prompt_type: str, prompt_id: str) -> str:
    """Call Anthropic Claude model and return text output."""
    global anthropic_client

    if MOCK_MODE:
        return mock_response("claude", prompt_type, prompt_id, prompt)

    if anthropic_client is None:
        try:
            anthropic_client = Anthropic()
        except Exception as e:
            return f"ERROR (Claude): {e}"

    for attempt in range(2):
        try:
            response = anthropic_client.messages.create(
                model=CLAUDE_MODEL_NAME,
                max_tokens=1200,
                messages=[
                    {"role": "user", "content": prompt}
                ],
            )

            text_parts = []
            for block in response.content:
                if block.type == "text":
                    text_parts.append(block.text)

            return "\n".join(text_parts).strip()
        except Exception as e:
            if attempt == 0:
                time.sleep(1)
            else:
                return f"ERROR (Claude): {e}"


print("=== Run configuration ===")
print(f"Project root: {PROJECT_ROOT}")
print(f"MOCK_MODE={MOCK_MODE}")
print(f"RUN_LIMIT={RUN_LIMIT}")
print(f"RUN_FULL_EXPERIMENT={RUN_FULL_EXPERIMENT}")
print(f"OpenAI model: {OPENAI_MODEL_NAME}")
print(f"Anthropic model: {CLAUDE_MODEL_NAME}")


# Step 8: Test on P001 first (fallback to first row if missing)
p001_rows = prompts_df[prompts_df["prompt_id"] == "P001"]
first_row = p001_rows.iloc[0] if not p001_rows.empty else prompts_df.iloc[0]

first_prompt_id = first_row["prompt_id"]
first_candidate = get_candidate_value(first_row)
first_job = first_row["job_description"]
first_task_instruction = get_task_instruction_value(first_row)

first_baseline_prompt = format_prompt(
    baseline_template,
    first_candidate,
    first_job,
    first_task_instruction,
)
first_context_prompt = format_prompt(
    context_template,
    first_candidate,
    first_job,
    first_task_instruction,
)

print("\n=== ONE-PROMPT TEST (P001 preferred): GPT baseline ===")
print(call_gpt(first_baseline_prompt, "baseline", first_prompt_id))
print("\n=== ONE-PROMPT TEST (P001 preferred): GPT context ===")
print(call_gpt(first_context_prompt, "context", first_prompt_id))
print("\n=== ONE-PROMPT TEST (P001 preferred): Claude baseline ===")
print(call_claude(first_baseline_prompt, "baseline", first_prompt_id))
print("\n=== ONE-PROMPT TEST (P001 preferred): Claude context ===")
print(call_claude(first_context_prompt, "context", first_prompt_id))


# Step 9: Select run set safely
if RUN_FULL_EXPERIMENT:
    prompts_to_run = prompts_df.copy()
else:
    safe_limit = 1 if RUN_LIMIT is None else max(1, int(RUN_LIMIT))
    if safe_limit == 1 and "P001" in set(prompts_df["prompt_id"]):
        prompts_to_run = prompts_df[prompts_df["prompt_id"] == "P001"].head(1).copy()
    else:
        prompts_to_run = prompts_df.head(safe_limit).copy()

print(f"\nRunning {len(prompts_to_run)} prompts out of {len(prompts_df)} total")


# Step 10: Loop through selected rows and collect outputs
results = []

for i, row in prompts_to_run.iterrows():
    print(f"Processing prompt {i + 1}/{len(prompts_to_run)} ({row['prompt_id']})")

    candidate_value = get_candidate_value(row)
    job_value = row["job_description"]
    task_instruction_value = get_task_instruction_value(row)

    baseline_prompt = format_prompt(
        baseline_template,
        candidate_value,
        job_value,
        task_instruction_value,
    )
    context_prompt = format_prompt(
        context_template,
        candidate_value,
        job_value,
        task_instruction_value,
    )

    scenario_type = row.get("scenario_type", "unknown")

    # GPT baseline
    gpt_baseline_output = call_gpt(baseline_prompt, "baseline", row["prompt_id"])
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "scenario_type": scenario_type,
            "model": "gpt",
            "prompt_type": "baseline",
            "output": gpt_baseline_output,
            "output_length": len(gpt_baseline_output),
        }
    )

    # GPT context
    gpt_context_output = call_gpt(context_prompt, "context", row["prompt_id"])
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "scenario_type": scenario_type,
            "model": "gpt",
            "prompt_type": "context",
            "output": gpt_context_output,
            "output_length": len(gpt_context_output),
        }
    )

    # Claude baseline
    claude_baseline_output = call_claude(baseline_prompt, "baseline", row["prompt_id"])
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "scenario_type": scenario_type,
            "model": "claude",
            "prompt_type": "baseline",
            "output": claude_baseline_output,
            "output_length": len(claude_baseline_output),
        }
    )

    # Claude context
    claude_context_output = call_claude(context_prompt, "context", row["prompt_id"])
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "scenario_type": scenario_type,
            "model": "claude",
            "prompt_type": "context",
            "output": claude_context_output,
            "output_length": len(claude_context_output),
        }
    )


# Step 11: Build results dataframe in expected schema order
expected_columns = [
    "prompt_id",
    "scenario_type",
    "model",
    "prompt_type",
    "output",
    "output_length",
]

results_df = pd.DataFrame(results, columns=expected_columns)


# Step 12: Save results only if there is at least one non-error output
if results_df.empty:
    print("\nNo rows generated. Skipping save.")
elif results_df["output"].apply(is_error_output).all():
    print("\nAll outputs are API errors. Skipping final CSV save for safety.")
else:
    results_df.to_csv(RESULTS_PATH, index=False)
    print(f"\nDone. Saved {len(results_df)} rows to: {RESULTS_PATH}")

print(results_df.head())


# Step 13: Quick sanity checks
print("\n=== Sanity checks ===")
print("Rows generated:", len(results_df))
print("Error outputs:", results_df["output"].apply(is_error_output).sum())
print("Empty outputs:", results_df["output"].fillna("").str.strip().eq("").sum())

expected_combos = {
    ("gpt", "baseline"),
    ("gpt", "context"),
    ("claude", "baseline"),
    ("claude", "context"),
}
missing_combo_count = 0
for prompt_id, group in results_df.groupby("prompt_id"):
    combos = set(zip(group["model"], group["prompt_type"]))
    if expected_combos - combos:
        missing_combo_count += 1
print("Prompts with missing model/prompt combinations:", missing_combo_count)
